# Plane-parallel slab - high-level `LineRt` with callable fields

This notebook mirrors `docs/examples/plane_parallel_hl.py` but in
interactive form.  A uniform slab of CO (J=1->0) is illuminated from
one face; photons scatter in the line and are continuously absorbed
by dust.  The field arrays (`n_species`, `temperature`, `vel`) are
provided as **callables** so they can be spatially varying.

Run directory is auto-created under `/tmp/line_rt/`.

In [ ]:
import sys, os
_PROJECT = os.path.expanduser('~/Seafile/seafile_sync/code/line_rt_pipeline')
if _PROJECT not in sys.path:
    sys.path.insert(0, _PROJECT)

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from core.line_rt import LineRt

AU = 1.49598e13   # cm
Lsun = 3.828e33   # erg/s

## 1. Callable field functions

Each callable receives `(X, Y, Z)` - a tuple of 3D `(nz, ny, nx)` arrays
of cell-centre coordinates in CGS [cm] - and returns a 3D array of the
same shape.  Here we use uniform fields for clarity, but any
spatially-varying function works.

In [ ]:
def n_total_callable(X, Y, Z):
    res = np.full(X.shape, n_species, dtype=np.float64)
    return res

def temperature_callable(X, Y, Z):
    return np.full(X.shape, temperature, dtype=np.float64)

def vx_callable(X, Y, Z):
    return np.zeros(X.shape, dtype=np.float64)

## 2. Configure `LineRt` (Group 1: species-based)

Pass the callables directly as `n_species`, `temperature`, and `vel`.
`LineRt` calls them internally after building the mesh.

In [ ]:
rt = LineRt(
    n_cell=(64, 2, 2),
    x_min=(-5, 0, 0), x_max=(5, 0.2, 0.2),
    unit_l0=AU, unit_t0=1.0,

    species='CO', transition_idx=0,
    n_species=n_total_callable,
    temperature=temperature_callable,
    vel=(vx_callable, 0.0, 0.0),
    mol_mass=28.0,

    ph_mode=2,            # R_IIA, const-mem (production mode)
    n_step=20000, n_scat=200000,
    n_cycles=3,
    n_emission_max=5,
    visualize=False,
)
rt.set_boundary('fre fre per per per per')

# Slab source at x=-5 (left face), 0.8 Lsun at 2.6 mm
rt.add_source(
    type='slab', x=-5.0, direction='+x',
    n_photon=20000,
    flux=0.8 * Lsun / (0.2 * 0.2 * AU * AU),
    wavelength=2.6e-1,    # CO J=1->0 ~ 115 GHz -> 2.6 mm [cm]
)

print(f'Mesh: {rt._n_cell}, sources: {len(rt._sources)}')

## 3. Run

The run directory is printed and everything (`.par`, `.bin` files)
goes under `/tmp/line_rt/rt_<timestamp>/`.

In [ ]:
results = rt.run()

## 4. Results

In [ ]:
res_list = results['results']
mesh = results['mesh']
nxc = int(mesh['n_cell'][0])
x_cell = np.linspace(-5, 5, nxc)

for k, res in enumerate(res_list):
    flx = res.get('flx')
    exc = res.get('exc_flux_flat', res.get('excitation_flux'))
    n_esc = len(res.get('photons', {}).get('vel', []))
    f_str = f'flx_max={np.max(flx):.2e}' if flx is not None else 'flx=N/A'
    e_str = f'exc_max={np.max(exc):.2e}' if exc is not None else 'exc=N/A'
    print(f'  Cycle {k}: {f_str}  {e_str}  n_esc={n_esc}')

### (a) Emergent spectrum

In [ ]:
spectrum = results.get('spectrum', {})
vel_data = np.asarray(spectrum.get('vel', []))

fig, ax = plt.subplots(figsize=(8, 4))
if len(vel_data) > 0:
    ax.hist(vel_data * 1e-5, bins=60, density=True, alpha=0.6, color='steelblue')
    ax.set(xlabel='v [km/s]', ylabel='PDF')
else:
    ax.text(0.5, 0.5, 'No escaped photons', transform=ax.transAxes, ha='center')
ax.set_title('Emergent Spectrum (CO J=1->0)')
plt.show()

### (b) Spatial flux profile

In [ ]:
exc_flat = results.get('exc_flux_flat')
flx_flat = results.get('flx')

fig, ax = plt.subplots(figsize=(8, 4))
if exc_flat is not None and flx_flat is not None:
    exc_x = exc_flat[:nxc]
    flx_x = flx_flat[:nxc]
    ax.plot(x_cell, exc_x / (exc_x.max() or 1), 'b-', lw=2, label='excitation_flux')
    ax.plot(x_cell, flx_x / (flx_x.max() or 1), color='orange', lw=1.5, label='flx')
    ax.set(xlabel='x [AU]', ylabel='normalised')
    ax.set_yscale('log')
    ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'No flux data', transform=ax.transAxes, ha='center')
ax.set_title('Spatial Flux Distribution')
plt.show()

### (c) Default multi-panel plot

`default_plot()` from `core.visualize` generates a 2-column, N-row
panel grid.  The default field list: emergent spectrum (km/s histogram),
flux map, mfp_i_sca_0, b_sca, excited fraction, emissivity - all as
logarithmic colormaps (except the spectrum).  Missing fields leave a
blank panel with title.  Pass `dyn_range=True` to clip the color scale
to integer-dex bounds.

In [ ]:
from core.visualize import default_plot

fig, axes = default_plot(results, dyn_range=False)

### (d) Population convergence

In [ ]:
pop_hist = [r['populations'] for r in res_list if 'populations' in r]

fig, ax = plt.subplots(figsize=(8, 4))
if len(pop_hist) >= 2:
    from core.visualize import plot_convergence
    plot_convergence(ax, pop_hist, list(range(len(pop_hist))))
else:
    ax.text(0.5, 0.5, 'No convergence data', transform=ax.transAxes, ha='center')
ax.set_title('Level Population Convergence')
plt.show()

## 5. Inspect the run directory

All temporary files (`.par`, `.bin`) live under `/tmp/line_rt/`.

In [ ]:
run_dir = results.get('run_dir') or rt._resolve_path()
print(f'Run directory: {run_dir}')
for f in sorted(os.listdir(run_dir)):
    sz = os.path.getsize(os.path.join(run_dir, f))
    print(f'  {f:30s}  {sz:>10d} bytes')